# Advanced RAG: Parent-Document Retrieval 🚀
### Part of the `RAG-Journey` Portfolio Series

## The Problem It Solves
Standard RAG pipelines often face a contradiction when splitting documents:
1. **Small Chunks:** Great for embedding models to capture precise semantic meaning, but the LLM loses the broader context, leading to incomplete answers.
2. **Large Chunks:** Great for providing rich context to the LLM, but the semantic meaning gets diluted, making it harder for the vector database to find exact matches.

## The Solution
The **Parent-Document Retriever** decouples the data stored for *retrieval* from the data passed to the *LLM*:
* **Child Chunks:** The document is split into small chunks (e.g., 128 tokens) and stored in the **Vector Database** for highly accurate semantic search.
* **Parent Chunks:** The document is also split into larger chunks (e.g., 512 tokens) and stored in an **In-Memory Document Store**, mapped to their corresponding children.

When a query is made, the system finds the relevant *child* chunk but passes its corresponding *parent* chunk to the LLM.

# Environment Setup & Fixed Imports

In [ ]:
# 1. Install required packages
!pip install --quiet langchain-community langchain-core faiss-cpu langchain-groq

# 2. Setup Kaggle Secrets for API Keys
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

# 3. Standard Imports (LCEL & Community Extensions)
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
# Using HuggingFace via LangChain community wrappers as built in your basics track
from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# For Parent-Child specifically:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

## Step 1: Initialize Mock Technical Data
We will use a highly contextual document about LLM architectures to clearly demonstrate how context is preserved or lost.

In [ ]:
# Creating sample data to demonstrate context preservation
docs = [
    Document(
        page_content="""The Gemini-1.5-Pro model introduced a revolutionary 2-million token context window. 
        This architecture utilizes an advanced Mixture-of-Experts (MoE) approach. 
        When scaling context, traditional attention mechanisms encounter an O(N^2) computational bottleneck. 
        To mitigate this, Gemini implements optimized attention kernels. 
        Engineers testing long-context retrieval found a near-perfect 100% recall accuracy in 'Needle In A Haystack' evaluations across the entire context field.""",
        metadata={"source": "gemini_deep_dive.txt", "category": "LLM Architecture"}
    )
]

## Step 2: Configure Parent & Child Splitters
* **Parent Splitter:** Creates large blocks (`chunk_size=512`) to keep the structural story intact for the LLM.
* **Child Splitter:** Breaks those blocks down into micro-chunks (`chunk_size=128`) for the vector store.

In [ ]:
# 1. Define the structural splitters
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=128, chunk_overlap=10)

# 2. Initialize the storage layer for Parent Documents
# The vector store only stores the tiny child chunks. The docstore holds the full parent text.
vectorstore = FAISS.from_documents([Document(page_content="init")], HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
store = InMemoryStore()

# 3. Instantiate the ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 4. Add the documents to the retriever (This automatically splits them into children and indexes them)
retriever.add_documents(docs, ids=None)

## Step 3: Verification (Vector Search vs. Parent Retrieval)
Let's see what happens when we ask a specific question. We will compare what the raw child vector search sees versus what our parent retriever fetches.

In [ ]:
query = "What is the recall accuracy in long-context evaluations?"

# Let's check what the vectorstore sees vs what the retriever returns
print("--- CHILD VECTOR STORE SEARCH RESULT ---")
child_results = vectorstore.similarity_search(query)
for i, doc in enumerate(child_results):
    print(f"Child {i+1}: {doc.page_content}\n")

print("\n--- PARENT RETRIEVER RESULT ---")
parent_results = retriever.invoke(query)
for i, doc in enumerate(parent_results):
    print(f"Parent {i+1}: {doc.page_content}\n")

## Step 4: Build the LCEL Chain with Groq
Now, we integrate the parent retriever into a standard LCEL chain using `llama-3.1-8b-instant`. The LLM will receive the complete, rich context block rather than just the single matched sentence.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 1. System Prompt Construction
template = """You are an elite AI Research Assistant. Answer the user's question using ONLY the provided parent context. 
If you do not know the answer based on the context, state that explicitly.

Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# 2. LLM Initialization
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# 3. LCEL Chain Construction
parent_rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Execution
response = parent_rag_chain.invoke(query)
print("=== FINAL GENERATION ===")
print(response)